In [1]:
##########################################
## Python/OpenRouter code to perform USMLE Med Question Answer
##
## Dataset comes from: https://github.com/chat-data-llc/medical_chat_performance_evaluation/blob/main/test_datasets/USMLE/Medical%20Chat%20USMLE%20Correctness%20Check%20-%20Test%201.csv
##
## Author: Christopher Meaney
## Date: September 2026
##########################################

In [2]:
##########
## Package dependencies
##########

In [3]:
## For connection to open router
from openai import OpenAI

## For connection to Open Router, CSV output, etc.
import os

## For dataframes and data wrangling
import pandas as pd
import numpy as np

## For timing
import time

## For JSON data structure
import json

## For regular expressions --- possibly needed to fix broken JSON structures returned by LLM
import re

In [4]:
## Pandas options
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 100)

In [5]:
##########
## File paths for input/output data
##########
WORK_DIR = r"/path/to/project/"

INPUT_EXAMS_CSV = os.path.join(
    WORK_DIR,
    "data",
    "Medical Chat USMLE Correctness Check - Test 1_CM.csv"
)

OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Checkpoint / run output
OUTPUT_CHECKPOINT_CSV = os.path.join(
    OUTPUT_DIR,
    "model_results.csv"
)

## Summary outputs
PERFORMANCE_SUMMARY_CSV = os.path.join(OUTPUT_DIR, "performance_summary.csv")

In [6]:
# help(pd.read_csv)

In [7]:
##########
## Import title abstract data
##########

In [8]:
df = pd.read_csv(INPUT_EXAMS_CSV, 
                 encoding="utf-8"
                )
df.head(n=6)

,Number,TypeQuestion,Question,CorrectResponse,ChatGPT_Output
0,1.1,MC-NJ,Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy vo...,(B) 120-140,"With a 95% confidence level, the true mean for the population from which this sample was drawn f..."
1,1.2,MC-J,Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy vo...,(B) 120-140,The standard error of the mean is the standard deviation of the sampling distribution of the mea...
2,1.3,OE,Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy vo...,(B) 120-140,The standard error of the mean is a measure of the variability of the sample mean. It is calcula...
3,2.1,MC-NJ,"A 48-year-old man dies suddenly of a cardiac arrhythmia. Six weeks ago, he was resuscitated from...",(E) Normal kidney,It is difficult to accurately answer this question without more information. Oliguric renal fail...
4,2.2,MC-J,"A 48-year-old man dies suddenly of a cardiac arrhythmia. Six weeks ago, he was resuscitated from...",(E) Normal kidney,It is difficult to say for certain what the examination of the patient's kidney at autopsy would...
5,2.3,OE,"A 48-year-old man dies suddenly of a cardiac arrhythmia. Six weeks ago, he was resuscitated from...",(E) Normal kidney,It is impossible for me to say with certainty what the result of the examination of the patient'...


In [9]:
## Dimensions of title/abstract corpus
df.shape

(271, 5)

In [10]:
## Names of columns
pd.Series(df.columns)

0             Number
1       TypeQuestion
2           Question
3    CorrectResponse
4     ChatGPT_Output
dtype: str

In [11]:
## Number of words (assuming white space tokenization) from USMLE/MCQ corpus
df["question_word_count"] = df["Question"].astype(str).str.split().str.len()

df[["question_word_count"]].describe()

,question_word_count
count,271.000000
mean,118.726937
std,46.518009
min,41.000000
25%,87.000000
50%,111.000000
75%,141.000000
max,285.000000


In [12]:
df[["question_word_count"]].quantile([0, 0.01, 0.025, 0.05, 0.1, 0.25, 0.50, 0.75, 0.90, 0.95, 0.975, 0.99, 1])

,question_word_count
0.000,41.00
0.010,42.00
0.025,49.75
0.050,56.00
0.100,66.00
0.250,87.00
0.500,111.00
0.750,141.00
0.900,184.00
0.950,206.50


In [13]:
## Type question
df.TypeQuestion.value_counts()

TypeQuestion
MC-NJ    94
MC-J     94
OE       83
Name: count, dtype: int64

In [14]:
df_ = df[df['TypeQuestion']=='MC-NJ']
df_.shape

(94, 6)

In [15]:
## Correct Response
df_["CorrectResponse_R"] = df_["CorrectResponse"].str.split().str[0].str.strip("()")
df_.CorrectResponse_R.value_counts().sort_index()

CorrectResponse_R
A    21
B    18
C    14
D    24
E    13
F     3
G     1
Name: count, dtype: int64

In [16]:
##########
## Randomly split/divide USMLE datset
##
## 1) 10 question "resevoir" used to select examples for prompting LLM
## 2) 84 question test dataset
##
## Note: We perform this split so no USMLE shot/example questions are ever used in test set evaluation
##########

In [17]:
# Reproducible dataset split
DATA_SPLIT_SEED = 12345
rng = np.random.default_rng(DATA_SPLIT_SEED)

In [18]:
## df_ is your 94-row MC-NJ dataframe
df_pool = df_.copy().reset_index(drop=True)

## Choose 10 reservoir questions
reservoir_idx = rng.choice(df_pool.index, size=10, replace=False)

## Split
df_reservoir = df_pool.loc[reservoir_idx].copy()
df_test = df_pool.drop(reservoir_idx).copy()

[df_reservoir.shape, df_test.shape]

[(10, 7), (84, 7)]

In [19]:
# reservoir_idx

In [20]:
#####################
## Initialize OpenRouter client
#####################

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY") ## Need to bash: export OPENROUTER_API_KEY=[your_key]

assert OPENROUTER_API_KEY is not None, (
    "OPENROUTER_API_KEY environment variable is not set."
)

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1", ## OpenRouter model endpoints
)

print("OpenRouter client initialized.")

OpenRouter client initialized.


In [21]:
###################################
##
## Prompt engineering, components:
##
## - Role:                          (None, Medical Expert)
## - Explanation / reasoning cue:   (None, Brief explanation in JSON with constraints)
## - Examples / shots:              (None/zero, one-shot, few-shot)
##
## 2*2*3 ==> 12 prompt templates to explore impacts of role, explanation, shots ==> under factorial design
##
###################################

In [22]:
##
## Helper function to select example/shots for inclusion in prompt
##
def make_shot_example(row):
    q = str(row["Question"]).strip()
    ans = str(row["CorrectResponse_R"]).strip()

    payload = {"answer": ans}

    payload_str = json.dumps(payload, ensure_ascii=False)

    template = """
Question:
<QUESTION>

Answer:
<PAYLOAD>
""".strip()

    return (
        template
        .replace("<QUESTION>", q)
        .replace("<PAYLOAD>", payload_str)
    )

In [23]:
# make_shot_example(row=df_reservoir.iloc[0])

In [24]:
##
## Helper function to sample shots from resevoir
##
def sample_shots(df_reservoir, n_shots, shot_rng=None):

    sampled = df_reservoir.sample(
        n=n_shots,
        replace=False,
        random_state=shot_rng,
    )

    shot_strings = [
        make_shot_example(row)
        for _, row in sampled.iterrows()
    ]

    return "\n\n".join(shot_strings)

In [25]:
# sample_shots(df_reservoir, n_shots=3, random_state=SEED)

In [26]:
##
## Define 12 types of prompts to expect in this experiment
##
PROMPTS = [
    {"name": "base_zero_noex",  "role": False, "explanation": False, "shots": 0},
    {"name": "base_one_noex",   "role": False, "explanation": False, "shots": 1},
    {"name": "base_few_noex",   "role": False, "explanation": False, "shots": 3},
    {"name": "base_zero_expl",  "role": False, "explanation": True,  "shots": 0},
    {"name": "base_one_expl",   "role": False, "explanation": True,  "shots": 1},
    {"name": "base_few_expl",   "role": False, "explanation": True,  "shots": 3},
    {"name": "role_zero_noex",  "role": True,  "explanation": False, "shots": 0},
    {"name": "role_one_noex",   "role": True,  "explanation": False, "shots": 1},
    {"name": "role_few_noex",   "role": True,  "explanation": False, "shots": 3},
    {"name": "role_zero_expl",  "role": True,  "explanation": True,  "shots": 0},
    {"name": "role_one_expl",   "role": True,  "explanation": True,  "shots": 1},
    {"name": "role_few_expl",   "role": True,  "explanation": True,  "shots": 3},
]

In [27]:
#####################
## Function to build prompt, based on MCQ, prompt specification (see above), shots/examples (from reservoir), and RNG
#####################
def build_messages(question, spec, df_reservoir, shot_rng):

    if spec["role"]:
        system_prompt = """
You are a medical expert answering USMLE-style multiple-choice questions.
Return only valid JSON.
""".strip()
    else:
        system_prompt = "Return only valid JSON."

    if spec["explanation"]:
        json_schema = '{"answer":"<A|B|C|D|E|F|G>","explanation":"<one short sentence>"}'
        rules_block = """
Rules:
- "answer" must be exactly one uppercase letter from A, B, C, D, E, F, or G.
- "explanation" must be exactly one sentence and no more than 25 words.
- Do not include any text outside the JSON object.
""".strip()
    else:
        json_schema = '{"answer":"<A|B|C|D|E|F|G>"}'
        rules_block = """
Rules:
- "answer" must be exactly one uppercase letter from A, B, C, D, E, F, or G.
- Do not include any text outside the JSON object.
""".strip()

    shots_block = ""
    if spec["shots"] > 0:
        sampled_shots = sample_shots(
            df_reservoir=df_reservoir,
            n_shots=spec["shots"],
            shot_rng=shot_rng
        )
        shots_block = f"""
Here are worked examples.

{sampled_shots}

Now answer the next question.
""".strip()

    user_prompt = f"""
Answer the question by selecting one option from A, B, C, D, E, F, or G.

Return exactly this JSON schema:
{json_schema}

{rules_block}

{shots_block}

Question:
{question}
""".strip()

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

In [28]:
##
## Loop over prompt specs (defined above) and investigate what the resulting fixed prompts might look like
##

In [29]:
test_question = str(df_test.iloc[0]["Question"])

DEMO_SHOT_SEED = 54321
demo_shot_rng = np.random.default_rng(DEMO_SHOT_SEED)

for spec in PROMPTS:
    print("=" * 80)
    print("PROMPT:", spec["name"])
    print("ROLE:", spec["role"], "EXPLANATION:", spec["explanation"], "SHOTS:", spec["shots"])

    messages = build_messages(
        question=test_question,
        spec=spec,
        df_reservoir=df_reservoir,
        shot_rng=demo_shot_rng,
    )

    print("\nSYSTEM:")
    print(messages[0]["content"])

    print("\nUSER:")
    print(messages[1]["content"])
    print("\n")

PROMPT: base_zero_noex
ROLE: False EXPLANATION: False SHOTS: 0

SYSTEM:
Return only valid JSON.

USER:
Answer the question by selecting one option from A, B, C, D, E, F, or G.

Return exactly this JSON schema:
{"answer":"<A|B|C|D|E|F|G>"}

Rules:
- "answer" must be exactly one uppercase letter from A, B, C, D, E, F, or G.
- Do not include any text outside the JSON object.



Question:
Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy volunteers. The data follow a normal distribution. The mean and standard deviation for this group are 130 mg/dL and 25 mg/dL, respectively. The standard error of the mean is 5.0. With a 95% confidence level, the true mean for the population from which this sample was drawn falls within which of the following ranges (in mg/dL)?

(A) 105-155
(B) 120-140
(C) 125-135
(D) 128-132
(E) 129-131


PROMPT: base_one_noex
ROLE: False EXPLANATION: False SHOTS: 1

SYSTEM:
Return only valid JSON.

USER:
Answer the question by s

In [30]:
###########
## Specify the LLM you want to answer USMLE questions
###########

In [31]:
MODEL_SPECS = [
    {
        "model": "deepseek/deepseek-v4-flash-0731",
        "use_system": True,
    }
]

MODEL = MODEL_SPECS[0]["model"]

MODEL_SPECS

[{'model': 'deepseek/deepseek-v4-flash-0731', 'use_system': True}]

In [32]:
MODEL_SPECS

[{'model': 'deepseek/deepseek-v4-flash-0731', 'use_system': True}]

In [33]:
MODEL

'deepseek/deepseek-v4-flash-0731'

In [34]:
##
## Specify/fix LLM model call configuration parameters
##

# Generation parameters
TEMPERATURE = 0.0
TOP_P = 1.0
DECODING_SEED = 202606
MAX_TOKENS = 256

# Reasoning configuration
REASONING_CONFIG = {
    "enabled": False,
}

# Output format
RESPONSE_FORMAT = {"type": "json_object"}

# OpenRouter provider configuration
PROVIDER_CONFIG = {
    "order": ["deepinfra"],
    "allow_fallbacks": False,
    "require_parameters": True,
}

# No web search or external retrieval tools are used
WEB_SEARCH_ENABLED = False

In [35]:
## Valid answers
VALID_ANSWERS = set("ABCDEFG")
VALID_ANSWERS

{'A', 'B', 'C', 'D', 'E', 'F', 'G'}

In [36]:
## Regex to repair (potentially) broken JSON structures
def try_repair_json(raw_text: str):
    if raw_text is None:
        return None

    text = str(raw_text).strip()

    # Remove markdown fences, e.g. ```json ... ```
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    # Extract the first JSON-looking object
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not match:
        return None

    candidate = match.group(0).strip()

    try:
        parsed = json.loads(candidate)
    except Exception:
        return None

    if not isinstance(parsed, dict):
        return None

    return parsed

In [37]:
#####################
## Submit one prompt to the LLM and parse the returned JSON response
##
## Transient HTTP 429 errors are retried up to two times.
## Provider fallback remains disabled.
#####################

def answer_one_from_messages(messages, model: str):
    t0 = time.time()
    usage = None
    actual_provider = None
    api_attempts = 0

    # Retry configuration for transient rate-limit errors
    MAX_API_ATTEMPTS = 3
    RETRY_DELAYS_SECONDS = [5, 10]

    # Record the generation and provider configuration associated with this request
    run_metadata = {
        "model": model,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "decoding_seed": DECODING_SEED,
        "max_tokens": MAX_TOKENS,
        "reasoning_enabled": REASONING_CONFIG["enabled"],
        "response_format": RESPONSE_FORMAT["type"],
        "requested_provider": PROVIDER_CONFIG["order"][0],
        "allow_fallbacks": PROVIDER_CONFIG["allow_fallbacks"],
        "require_parameters": PROVIDER_CONFIG["require_parameters"],
        "web_search_enabled": WEB_SEARCH_ENABLED,
    }

    # -------------------------------------------------
    # API request with retry for HTTP 429 only
    # -------------------------------------------------
    for attempt in range(1, MAX_API_ATTEMPTS + 1):

        api_attempts = attempt

        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                seed=DECODING_SEED,
                max_tokens=MAX_TOKENS,
                response_format=RESPONSE_FORMAT,
                extra_body={
                    "provider": PROVIDER_CONFIG,
                    "reasoning": REASONING_CONFIG,
                },
            )

            usage = getattr(resp, "usage", None)

            # Capture the provider that actually served the request
            actual_provider = getattr(resp, "provider", None)

            break

        except Exception as e:

            # Identify HTTP 429 rate-limit/provider-capacity errors
            status_code = getattr(e, "status_code", None)

            if status_code == 429 and attempt < MAX_API_ATTEMPTS:
                time.sleep(RETRY_DELAYS_SECONDS[attempt - 1])
                continue

            # Non-retryable error, or final failed retry attempt
            latency = time.time() - t0

            return {
                **run_metadata,
                "actual_provider": None,
                "api_attempts": api_attempts,
                "answer": None,
                "explanation": None,
                "status": "error",
                "failure_reason": "api_error",
                "error_detail": str(e),
                "latency_seconds": round(latency, 3),
                "prompt_tokens": None,
                "completion_tokens": None,
                "total_tokens": None,
                "raw_json": None,
                "repaired_json": False,
            }

    # -------------------------------------------------
    # Process successful API response
    # -------------------------------------------------
    latency = time.time() - t0

    if resp is None or not getattr(resp, "choices", None):
        return {
            **run_metadata,
            "actual_provider": actual_provider,
            "api_attempts": api_attempts,
            "answer": None,
            "explanation": None,
            "status": "error",
            "failure_reason": "no_choices_returned",
            "error_detail": None,
            "latency_seconds": round(latency, 3),
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
            "raw_json": None,
            "repaired_json": False,
        }

    message = resp.choices[0].message
    raw_text = getattr(message, "content", None)

    if raw_text is None or str(raw_text).strip() == "":
        return {
            **run_metadata,
            "actual_provider": actual_provider,
            "api_attempts": api_attempts,
            "answer": None,
            "explanation": None,
            "status": "error",
            "failure_reason": "no_message_content",
            "error_detail": None,
            "latency_seconds": round(latency, 3),
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
            "raw_json": raw_text,
            "repaired_json": False,
        }

    repaired_json = False

    # -------------------------------------------------
    # Parse JSON response
    # -------------------------------------------------
    try:
        parsed = json.loads(raw_text)

    except Exception as e:
        parsed = try_repair_json(raw_text)

        if parsed is None:
            return {
                **run_metadata,
                "actual_provider": actual_provider,
                "api_attempts": api_attempts,
                "answer": None,
                "explanation": None,
                "status": "error",
                "failure_reason": "json_parse_failed",
                "error_detail": str(e),
                "latency_seconds": round(latency, 3),
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
                "raw_json": raw_text,
                "repaired_json": False,
            }

        repaired_json = True

    # -------------------------------------------------
    # Validate answer fields
    # -------------------------------------------------
    answer = parsed.get("answer")
    explanation = parsed.get("explanation")

    if isinstance(answer, str):
        answer = answer.strip()
    else:
        answer = str(answer).strip() if answer is not None else None

    if not isinstance(explanation, str):
        explanation = None if explanation is None else str(explanation)

    if answer not in VALID_ANSWERS:
        return {
            **run_metadata,
            "actual_provider": actual_provider,
            "api_attempts": api_attempts,
            "answer": None,
            "explanation": explanation,
            "status": "error",
            "failure_reason": "answer_not_in_valid_set",
            "error_detail": f"Returned answer: {answer}",
            "latency_seconds": round(latency, 3),
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
            "raw_json": raw_text,
            "repaired_json": repaired_json,
        }

    # -------------------------------------------------
    # Successful result
    # -------------------------------------------------
    return {
        **run_metadata,
        "actual_provider": actual_provider,
        "api_attempts": api_attempts,
        "answer": answer,
        "explanation": explanation,
        "status": "ok",
        "failure_reason": None,
        "error_detail": None,
        "latency_seconds": round(latency, 3),
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
        "raw_json": raw_text,
        "repaired_json": repaired_json,
    }

In [38]:
#####################
## Test one complete prompt -> API -> parsing workflow
#####################

## Select model
model = MODEL

## Select prompt style
spec = PROMPTS[9]

## Select question
question_ = str(df_test.iloc[1]["Question"])

## Define reproducible demonstration-shot RNG
DEMO_SHOT_SEED = 54321
demo_shot_rng = np.random.default_rng(DEMO_SHOT_SEED)

## Construct prompt/messages to send to model
messages = build_messages(
    question=question_,
    spec=spec,
    df_reservoir=df_reservoir,
    shot_rng=demo_shot_rng,
)

## Send request to model
gen_text = answer_one_from_messages(
    messages=messages,
    model=model,
)

## Display result
gen_text

{'model': 'deepseek/deepseek-v4-flash-0731',
 'temperature': 0.0,
 'top_p': 1.0,
 'decoding_seed': 202606,
 'max_tokens': 256,
 'reasoning_enabled': False,
 'response_format': 'json_object',
 'requested_provider': 'deepinfra',
 'allow_fallbacks': False,
 'require_parameters': True,
 'web_search_enabled': False,
 'actual_provider': 'DeepInfra',
 'api_attempts': 1,
 'answer': 'C',
 'explanation': 'The patient had acute tubular necrosis, which typically heals by regeneration, but if severe, can leave fibrous scars.',
 'status': 'ok',
 'failure_reason': None,
 'error_detail': None,
 'latency_seconds': 1.662,
 'prompt_tokens': 298,
 'completion_tokens': 36,
 'total_tokens': 334,
 'raw_json': '{"answer": "C", "explanation": "The patient had acute tubular necrosis, which typically heals by regeneration, but if severe, can leave fibrous scars."}',
 'repaired_json': False}

In [39]:
##
## Test the 12 different prompts for a fixed model/question
##

model = MODEL
question_ = str(df_test.iloc[1]["Question"])

DEMO_SHOT_SEED = 65432
demo_shot_rng = np.random.default_rng(DEMO_SHOT_SEED)

for spec in PROMPTS:
    messages = build_messages(
        question=question_,
        spec=spec,
        df_reservoir=df_reservoir,
        shot_rng=demo_shot_rng,
    )

    out = answer_one_from_messages(
        messages=messages,
        model=model,
    )

    print("=" * 80)
    print("PROMPT:", spec["name"])
    print("RUNTIME:", out["latency_seconds"], "seconds")
    print("ANSWER:", out["answer"])
    print("STATUS:", out["status"])
    print("PROVIDER:", out["actual_provider"])
    print("RAW:", out["raw_json"])
    print()

PROMPT: base_zero_noex
RUNTIME: 0.387 seconds
ANSWER: E
STATUS: ok
PROVIDER: DeepInfra
RAW: {"answer": "E"}

PROMPT: base_one_noex
RUNTIME: 1.356 seconds
ANSWER: E
STATUS: ok
PROVIDER: DeepInfra
RAW: {"answer": "E"}

PROMPT: base_few_noex
RUNTIME: 0.559 seconds
ANSWER: E
STATUS: ok
PROVIDER: DeepInfra
RAW: {"answer": "E"}

PROMPT: base_zero_expl
RUNTIME: 1.944 seconds
ANSWER: E
STATUS: ok
PROVIDER: DeepInfra
RAW: {"answer": "E", "explanation": "The patient's renal function fully recovered, indicating healing without residual scarring, so the kidneys would appear normal at autopsy."}

PROMPT: base_one_expl
RUNTIME: 1.39 seconds
ANSWER: E
STATUS: ok
PROVIDER: DeepInfra
RAW: {"answer": "E", "explanation": "The patient's renal function fully recovered after acute tubular necrosis, so autopsy would show normal kidneys."}

PROMPT: base_few_expl
RUNTIME: 1.156 seconds
ANSWER: E
STATUS: ok
PROVIDER: DeepInfra
RAW: {"answer": "E", "explanation": "The patient's renal function fully recovered aft

In [40]:
##########################
## Loop over models and questions, collecting results
##########################

In [41]:
## For testing --- restrict to smaller dataset

# df_test = df_test.head(5).copy()
# df_test.shape


In [42]:
# -------------------------------------------------
# Results file setup
# -------------------------------------------------
RESULT_COLUMNS = [
    # Prompt / experimental factors
    "prompt_name",
    "role",
    "explanation",
    "shots",
    
    # Question
    "question_id",
    "question",
    "correct_response",
    
    # Model / generation configuration
    "model",
    "temperature",
    "top_p",
    "decoding_seed",
    "max_tokens",
    "reasoning_enabled",
    "response_format",
    
    # Provider / tool configuration
    "requested_provider",
    "actual_provider",
    "allow_fallbacks",
    "require_parameters",
    "web_search_enabled",
    
    # API execution / retry information
    "api_attempts",
    
    # Model response
    "answer",
    "status",
    "failure_reason",
    "error_detail",
    "latency_seconds",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "raw_json",
    "repaired_json",
]

if not os.path.exists(OUTPUT_CHECKPOINT_CSV):
    pd.DataFrame(columns=RESULT_COLUMNS).to_csv(
        OUTPUT_CHECKPOINT_CSV,
        index=False
    )


# -------------------------------------------------
# Run evaluation loop: PROMPTS x Questions
# -------------------------------------------------
results = []
t0 = time.time()

# Fixed model for this experiment
model = MODEL

# Fixed RNG for reproducible in-context example sampling
SHOT_SAMPLING_SEED = 202606
shot_rng = np.random.default_rng(SHOT_SAMPLING_SEED)

for spec in PROMPTS:

    print(f"\nRunning prompt: {spec['name']}")

    for i, row in df_test.iterrows():

        question_ = str(row["Question"])
        question_id_ = row["Number"] if "Number" in df_test.columns else i
        correct_response_ = str(row["CorrectResponse_R"]).strip()

        try:
            messages = build_messages(
                question=question_,
                spec=spec,
                df_reservoir=df_reservoir,
                shot_rng=shot_rng,
            )

            out = answer_one_from_messages(
                messages=messages,
                model=model,
            )

            result_row = {
                # Prompt / experimental factors
                "prompt_name": spec["name"],
                "role": spec["role"],
                "explanation": spec["explanation"],
                "shots": spec["shots"],

                # Question
                "question_id": question_id_,
                "question": question_,
                "correct_response": correct_response_,

                # Model / generation configuration
                "model": out["model"],
                "temperature": out["temperature"],
                "top_p": out["top_p"],
                "decoding_seed": out["decoding_seed"],
                "max_tokens": out["max_tokens"],
                "reasoning_enabled": out["reasoning_enabled"],
                "response_format": out["response_format"],

                # Provider / tool configuration
                "requested_provider": out["requested_provider"],
                "actual_provider": out["actual_provider"],
                "allow_fallbacks": out["allow_fallbacks"],
                "require_parameters": out["require_parameters"],
                "web_search_enabled": out["web_search_enabled"],

                # API execution / retry information
                "api_attempts": out["api_attempts"],

                # Model response
                "answer": out["answer"],
                "status": out["status"],
                "failure_reason": out["failure_reason"],
                "error_detail": out["error_detail"],
                "latency_seconds": out["latency_seconds"],
                "prompt_tokens": out["prompt_tokens"],
                "completion_tokens": out["completion_tokens"],
                "total_tokens": out["total_tokens"],
                "raw_json": out["raw_json"],
                "repaired_json": out["repaired_json"],
            }

        except Exception as e:

            result_row = {
                # Prompt / experimental factors
                "prompt_name": spec["name"],
                "role": spec["role"],
                "explanation": spec["explanation"],
                "shots": spec["shots"],

                # Question
                "question_id": question_id_,
                "question": question_,
                "correct_response": correct_response_,

                # Model / generation configuration
                "model": model,
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "decoding_seed": DECODING_SEED,
                "max_tokens": MAX_TOKENS,
                "reasoning_enabled": REASONING_CONFIG["enabled"],
                "response_format": RESPONSE_FORMAT["type"],

                # Provider / tool configuration
                "requested_provider": PROVIDER_CONFIG["order"][0],
                "actual_provider": None,
                "allow_fallbacks": PROVIDER_CONFIG["allow_fallbacks"],
                "require_parameters": PROVIDER_CONFIG["require_parameters"],
                "web_search_enabled": WEB_SEARCH_ENABLED,

                # API execution / retry information
                "api_attempts": None,

                # Failure information
                "answer": None,
                "status": "exception",
                "failure_reason": "loop_exception",
                "error_detail": str(e),
                "latency_seconds": None,
                "prompt_tokens": None,
                "completion_tokens": None,
                "total_tokens": None,
                "raw_json": None,
                "repaired_json": False,
            }

        results.append(result_row)

        pd.DataFrame([result_row]).to_csv(
            OUTPUT_CHECKPOINT_CSV,
            mode="a",
            header=False,
            index=False,
        )

        time.sleep(1.0)

t1 = time.time()
runtime = t1 - t0

print("Total runtime (seconds):", round(runtime, 3))


Running prompt: base_zero_noex

Running prompt: base_one_noex

Running prompt: base_few_noex

Running prompt: base_zero_expl

Running prompt: base_one_expl

Running prompt: base_few_expl

Running prompt: role_zero_noex

Running prompt: role_one_noex

Running prompt: role_few_noex

Running prompt: role_zero_expl

Running prompt: role_one_expl

Running prompt: role_few_expl
Total runtime (seconds): 2916.644


In [43]:
results_df = pd.DataFrame(results)
results_df.shape

(1008, 30)

In [44]:
runtime

2916.644080877304

In [45]:
runtime/60/60

0.8101789113548067

In [46]:
len(results)

1008

In [47]:
pd.DataFrame({'vars': results_df.columns})

,vars
0,prompt_name
1,role
2,explanation
3,shots
4,question_id
5,question
6,correct_response
7,model
8,temperature
9,top_p


In [48]:
results_df.head(2)

,prompt_name,role,explanation,shots,question_id,question,correct_response,model,temperature,top_p,decoding_seed,max_tokens,reasoning_enabled,response_format,requested_provider,actual_provider,allow_fallbacks,require_parameters,web_search_enabled,api_attempts,answer,status,failure_reason,error_detail,latency_seconds,prompt_tokens,completion_tokens,total_tokens,raw_json,repaired_json
0,base_zero_noex,False,False,0,1.1,Serum LDL-cholesterol concentrations are maeasured in blood samples collected from 25 healthy vo...,B,deepseek/deepseek-v4-flash-0731,0.0,1.0,202606,256,False,json_object,deepinfra,DeepInfra,False,True,False,1,B,ok,NaN,NaN,0.974,225.0,7.0,232.0,"{""answer"": ""B""}",False
1,base_zero_noex,False,False,0,2.1,"A 48-year-old man dies suddenly of a cardiac arrhythmia. Six weeks ago, he was resuscitated from...",E,deepseek/deepseek-v4-flash-0731,0.0,1.0,202606,256,False,json_object,deepinfra,DeepInfra,False,True,False,1,E,ok,NaN,NaN,1.437,257.0,7.0,264.0,"{""answer"": ""E""}",False


In [49]:
############
## Summarize predictions: i.e. what titles/abstracts are incl-vs-excl from review
############

In [50]:
## Determine if model response matches correct answer (from key)
results_df["is_correct"] = results_df["answer"] == results_df["correct_response"]

In [51]:
## Model
results_df.model.value_counts()

model
deepseek/deepseek-v4-flash-0731    1008
Name: count, dtype: int64

In [52]:
##
## Inspect failure reasons
##

In [53]:
results_df["status"].value_counts(dropna=False)

status
ok       1000
error       8
Name: count, dtype: int64

In [54]:
results_df["failure_reason"].value_counts(dropna=False)

failure_reason
NaN          1000
api_error       8
Name: count, dtype: int64

In [55]:
## Number of valid runs/returns for each model (i.e. non failing outputs)
n_valid_df = (
    results_df
    .groupby("model")["status"]
    .apply(lambda x: (x == "ok").sum())
    .reset_index(name="n_valid")
)

print(n_valid_df)

                             model  n_valid
0  deepseek/deepseek-v4-flash-0731     1000


In [56]:
results_df["is_valid"] = results_df["status"] == "ok"

In [57]:
#########################
## MCQ Accuracy table
#########################

In [58]:
summary_df = (
    results_df
    .groupby(["prompt_name", "role", "explanation", "shots"])
    .agg(
        n=("question_id", "size"),
        n_ok=("is_valid", "sum"),
        n_correct=("is_correct", "sum"),
        avg_latency_seconds=("latency_seconds", "mean"),
        avg_prompt_tokens=("prompt_tokens", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        repaired_rate=("repaired_json", "mean"),
        retry_rate=("api_attempts", lambda x: (x > 1).mean()),
    )
    .reset_index()
)

summary_df["n_invalid"] = (
    summary_df["n"] - summary_df["n_ok"]
)

summary_df["ok_rate"] = (
    summary_df["n_ok"] / summary_df["n"]
)

summary_df["strict_acc"] = (
    summary_df["n_correct"] / summary_df["n"]
)

summary_df["cond_acc"] = np.where(
    summary_df["n_ok"] > 0,
    summary_df["n_correct"] / summary_df["n_ok"],
    np.nan
)

summary_df = (
    summary_df
    .sort_values(["role", "explanation", "shots"])
)

#summary_df

In [59]:
summary_df = summary_df[
    [
        "prompt_name",
        "role",
        "explanation",
        "shots",
        "n",
        "n_ok",
        "n_invalid",
        "ok_rate",
        "n_correct",
        "strict_acc",
        "cond_acc",
        "retry_rate",
        "repaired_rate",
        "avg_latency_seconds",
        "avg_prompt_tokens",
        "avg_completion_tokens",
        "avg_total_tokens",
    ]
]

summary_df

,prompt_name,role,explanation,shots,n,n_ok,n_invalid,ok_rate,n_correct,strict_acc,cond_acc,retry_rate,repaired_rate,avg_latency_seconds,avg_prompt_tokens,avg_completion_tokens,avg_total_tokens
5,base_zero_noex,False,False,0,84,84,0,1.000000,76,0.904762,0.904762,0.000000,0.0,0.811024,284.428571,7.000000,291.428571
3,base_one_noex,False,False,1,84,84,0,1.000000,75,0.892857,0.892857,0.011905,0.0,1.209321,459.892857,7.000000,466.892857
1,base_few_noex,False,False,3,84,84,0,1.000000,76,0.904762,0.904762,0.011905,0.0,1.219012,786.928571,7.000000,793.928571
4,base_zero_expl,False,True,0,84,84,0,1.000000,77,0.916667,0.916667,0.000000,0.0,1.763274,311.428571,36.904762,348.333333
2,base_one_expl,False,True,1,84,84,0,1.000000,75,0.892857,0.892857,0.011905,0.0,1.668167,481.380952,36.619048,518.000000
0,base_few_expl,False,True,3,84,83,1,0.988095,76,0.904762,0.915663,0.095238,0.0,3.564607,809.180723,37.048193,846.228916
11,role_zero_noex,True,False,0,84,84,0,1.000000,77,0.916667,0.916667,0.000000,0.0,1.054488,298.428571,7.000000,305.428571
9,role_one_noex,True,False,1,84,84,0,1.000000,75,0.892857,0.892857,0.011905,0.0,1.204964,481.321429,7.000000,488.321429
7,role_few_noex,True,False,3,84,82,2,0.976190,76,0.904762,0.926829,0.083333,0.0,1.834250,808.085366,7.000000,815.085366
10,role_zero_expl,True,True,0,84,84,0,1.000000,78,0.928571,0.928571,0.000000,0.0,1.892750,325.428571,36.500000,361.928571


In [60]:
summary_df.to_csv(PERFORMANCE_SUMMARY_CSV, index=False)

In [61]:
###################
## Properties of Jupyter Notebook env
####################

In [62]:
import sys
import platform
import datetime

In [63]:
##
## Session information
##
print("=== Session Info ===")
print("Execution date/time:", datetime.datetime.now().astimezone())
print("Python version:", sys.version)
print("Platform:", platform.platform())

=== Session Info ===
Execution date/time: 2026-09-17 11:51:37.335269-04:00
Python version: 3.14.3 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 22:51:43) [GCC 14.3.0]
Platform: Linux-4.4.0-26100-Microsoft-x86_64-with-glibc2.31


In [64]:
##
## Package versions
##
print("=== Key Packages ===")

import openai
import requests

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("openai:", openai.__version__)
print("requests:", requests.__version__)

=== Key Packages ===
pandas: 3.0.1
numpy: 2.4.3
openai: 2.14.0
requests: 2.32.5


In [65]:
##
## Experimental configuration
##

print("=== Experimental Configuration ===")

print("Model:", MODEL)
print("Temperature:", TEMPERATURE)
print("Top-p:", TOP_P)
print("Decoding seed:", DECODING_SEED)
print("Max tokens:", MAX_TOKENS)
print("Reasoning enabled:", REASONING_CONFIG["enabled"])
print("Response format:", RESPONSE_FORMAT["type"])

print("Requested provider:", PROVIDER_CONFIG["order"][0])
print("Provider fallbacks:", PROVIDER_CONFIG["allow_fallbacks"])
print("Require parameter support:", PROVIDER_CONFIG["require_parameters"])
print("Web search enabled:", WEB_SEARCH_ENABLED)

print("Dataset split seed:", DATA_SPLIT_SEED)
print("Shot sampling seed:", SHOT_SAMPLING_SEED)

=== Experimental Configuration ===
Model: deepseek/deepseek-v4-flash-0731
Temperature: 0.0
Top-p: 1.0
Decoding seed: 202606
Max tokens: 256
Reasoning enabled: False
Response format: json_object
Requested provider: deepinfra
Provider fallbacks: False
Require parameter support: True
Web search enabled: False
Dataset split seed: 12345
Shot sampling seed: 202606
